# SAGE Evaluation Metrics Verification & Protocol Analysis

**Tác giả:** Lộc lớn (Evaluation Owner)
**Mục tiêu:** Kiểm nghiệm toán học và thực nghiệm cho các chỉ số đánh giá hợp nhất ảnh IVIF:
- **EN (Entropy)**: Lượng thông tin rời rạc.
- **SD (Standard Deviation)**: Độ tương phản toàn ảnh theo quy ước population standard deviation.
- **SCD (Sum of Correlation Differences)**: Độ tương quan chênh lệch khử thông tin bổ trợ.
- **MS-SSIM (Multi-Scale Structural Similarity)**: Bảo toàn cấu trúc đa tỷ lệ và phân tích cơ chế tổng hợp đa nguồn (Mean vs Sum vs Variance-weighted).


In [ ]:
import numpy as np
import pandas as pd

# Nhập trực tiếp các module do nhóm xây dựng trong src.evaluation
from src.evaluation.metrics import (
    entropy,
    standard_deviation,
    sum_of_correlation_differences,
    fusion_ms_ssim,
)
from src.evaluation.evaluator import FusionEvaluator

print("Module evaluation imported successfully.")


## 1. Kiểm nghiệm trên các ca đối chứng toán học (Controlled Reference Cases)

Để đảm bảo tính đúng đắn toán học, ta kiểm nghiệm trên 8 ca nhân tạo có tính chất lý thuyết xác định:
1. **Uniform Noise**: Nhiễu độc lập giữa IR và VIS.
2. **Constant Images**: Ảnh hằng (toàn bộ 128) -> EN = 0.0, SD = 0.0, MS-SSIM = 1.0.
3. **Identity IR (F = IR)**: Fused chỉ sao chép IR -> MS-SSIM_IR = 1.0.
4. **Inverted IR (F = 255 - IR)**: Ngược pha hoàn toàn -> SCD âm.
5. **Linear 50/50 Mixture**: Trộn đều 0.5 IR + 0.5 VIS.
6. **Shifted Mixture (+30)**: Chứng minh tính bất biến tuyệt đối của SCD khi cộng hằng số vào F.
7. **High Contrast VIS Dominant**: VIS tương phản mạnh, IR tương phản yếu.
8. **Thermal Hotspot Salient**: Vùng nhiệt nổi bật trong IR được bảo toàn bằng Max fusion.


In [ ]:
rng = np.random.default_rng(2026)

cases = []
# Case 1
ir1 = rng.integers(0, 256, (180, 180), dtype=np.uint8)
vis1 = rng.integers(0, 256, (180, 180), dtype=np.uint8)
f1 = ((ir1.astype(float) + vis1.astype(float)) / 2).astype(np.uint8)
cases.append(("Uniform_Random_Mixture", f1, ir1, vis1))

# Case 2
ir2 = np.full((180, 180), 128, dtype=np.uint8)
vis2 = np.full((180, 180), 128, dtype=np.uint8)
f2 = np.full((180, 180), 128, dtype=np.uint8)
cases.append(("Constant_Images", f2, ir2, vis2))

# Case 3
ir3 = rng.integers(20, 230, (180, 180), dtype=np.uint8)
vis3 = rng.integers(20, 230, (180, 180), dtype=np.uint8)
f3 = ir3.copy()
cases.append(("Identity_IR_Only", f3, ir3, vis3))

# Case 4
ir4 = rng.integers(20, 230, (180, 180), dtype=np.uint8)
vis4 = rng.integers(20, 230, (180, 180), dtype=np.uint8)
f4 = (255 - ir4).astype(np.uint8)
cases.append(("Inverted_IR_Fusion", f4, ir4, vis4))

# Case 5 & 6
ir5 = rng.integers(30, 220, (180, 180), dtype=np.uint8)
vis5 = rng.integers(30, 220, (180, 180), dtype=np.uint8)
f5 = ((ir5.astype(float) + vis5.astype(float)) * 0.5).astype(np.uint8)
f6 = np.clip(f5.astype(float) + 30.0, 0, 255).astype(np.uint8)
cases.append(("Linear_50_50_Mixture", f5, ir5, vis5))
cases.append(("Shifted_Mixture_Add30", f6, ir5, vis5))

# Case 7 & 8
ir7 = rng.integers(120, 136, (180, 180), dtype=np.uint8)
vis7 = rng.integers(0, 256, (180, 180), dtype=np.uint8)
f7 = ((ir7.astype(float) + vis7.astype(float)) * 0.5).astype(np.uint8)
cases.append(("High_Contrast_VIS_Dominant", f7, ir7, vis7))

ir8 = np.full((180, 180), 30, dtype=np.uint8)
ir8[60:120, 60:120] = 230
vis8 = rng.integers(50, 100, (180, 180), dtype=np.uint8)
f8 = np.maximum(ir8, vis8)
cases.append(("Thermal_Hotspot_Salient_Max", f8, ir8, vis8))

rows = []
for name, f, a, b in cases:
    res = {
        "case_name": name,
        "EN": entropy(f),
        "SD": standard_deviation(f),
        "SCD": sum_of_correlation_differences(f, a, b),
    }
    ms = fusion_ms_ssim(f, a, b)
    res.update(ms)
    rows.append(res)

df_ref = pd.DataFrame(rows)
print(df_ref[["case_name", "EN", "SD", "SCD", "MS_SSIM_mean", "MS_SSIM_sum"]])


## 2. Phân tích cơ chế tổng hợp đa nguồn của MS-SSIM

Trong bài toán IVIF, có hai nguồn ảnh khác nhau (IR và VIS). Có 3 cách tổng hợp MS-SSIM chính:
1. **Equal Mean:** bash.5 	imes MS(F, IR) + 0.5 	imes MS(F, VIS)$, nằm trong 1$.
2. **Direct Sum:** (F, IR) + MS(F, VIS)$, nằm trong 2$.
3. **Variance-Weighted:** Trọng số tỷ lệ thuận với năng lượng/phương sai cục bộ của từng nguồn.


In [ ]:
agg_rows = []
for r in rows:
    ms_ir = r["MS_SSIM_IR"]
    ms_vis = r["MS_SSIM_VIS"]
    mean_val = 0.5 * (ms_ir + ms_vis)
    sum_val = ms_ir + ms_vis
    agg_rows.append({
        "case_name": r["case_name"],
        "MS_SSIM_IR": ms_ir,
        "MS_SSIM_VIS": ms_vis,
        "Equal_Mean": mean_val,
        "Direct_Sum": sum_val,
    })

df_agg = pd.DataFrame(agg_rows)
print(df_agg)


## 3. Khởi tạo Evaluator và Kiểm tra Pipeline Output Contract

Kiểm tra khả năng sinh các artifact chuẩn theo :
- : Từng ảnh có chỉ số, trạng thái hợp lệ, lý do lỗi nếu có.
- : Thống kê tổng hợp (mean, std, median, valid_count).


In [ ]:
evaluator = FusionEvaluator()
sample_record = evaluator.evaluate_arrays(f5, ir5, vis5, image_id="demo_sample")
print("Single image evaluation record:")
for k, v in sample_record.items():
    print(f"  {k}: {v}")
